# GNR 638 Project 2 — Visual MCQ Solver

Author: Rishabh Kumar (24b2419) · Partner: Dhruva Reddy

This notebook is the offline inference entrypoint. It reads the test set from `PARENT_DIR`, runs Qwen2.5-VL-7B with self-consistency decoding, and writes `submission.csv` matching the schema of `sample_submission.csv` (two columns: `image_name`, `option`).

**Pipeline summary**

1. Load Qwen2.5-VL-7B-Instruct in **bf16** (the model's native precision, ~15 GB VRAM) from the local `weights/` directory.
2. For each image in `test.csv`, sample N=7 reasoning chains at temperature 0.7.
3. Parse each sample's final answer digit (iron-clad parser; output always in `{1, 2, 3, 4, 5}`).
4. Majority-vote the seven samples; on ties involving skip (5) prefer the option vote.
5. Write `submission.csv`.

**Safety net**: a startup smoke test runs a single greedy generation on one image and verifies the parsed answer is in `{1..5}`. If the smoke test fails (load error or out-of-range output), `submit.py` errors out before processing all 50 questions — better to fail fast than to write a bad `submission.csv`.

**Expected runtime on the eval L40s**: ~25–35 min for 50 questions. Comfortably inside the 1 hr cap.

---

**TA: only one thing to set** — `PARENT_DIR` in the next cell. It must point at the directory containing `images/`, `test.csv`, and `sample_submission.csv`. Then run all cells (Cell → Run All).

In [ ]:
# === Configuration ===
# Set PARENT_DIR to the absolute path of the dir containing images/ and test.csv.
# Default assumes test data sits next to this notebook.
from pathlib import Path

PARENT_DIR = Path(".").resolve()                    # <-- adjust if test data lives elsewhere
PROJECT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path(".").resolve()
WEIGHTS_DIR = PROJECT_DIR / "weights" / "qwen2.5-vl-7b"
OUTPUT_CSV = PARENT_DIR / "submission.csv"

# Sanity checks - fail fast with clear messages if anything's wrong.
import sys
for path, label in [
    (PARENT_DIR / "images",   "images directory"),
    (PARENT_DIR / "test.csv", "test.csv"),
    (WEIGHTS_DIR,             "model weights (7B)"),
    (PROJECT_DIR / "src" / "submit.py", "src/submit.py"),
]:
    if not path.exists():
        sys.exit(f"ERROR: {label} not found at {path}. See README for layout.")

print(f"PARENT_DIR  : {PARENT_DIR}")
print(f"PROJECT_DIR : {PROJECT_DIR}")
print(f"WEIGHTS_DIR : {WEIGHTS_DIR}")
print(f"OUTPUT_CSV  : {OUTPUT_CSV}")

In [ ]:
# === Run inference ===
# Wraps src/submit.py with the production self-consistency configuration:
#   - bf16 model precision (the model's native; uses ~15 GB on the L40s)
#   - N=7 samples per question for majority voting
#   - temperature 0.7 for reasoning-chain diversity
#   - tau=0 (no confidence threshold); skip is only emitted when the majority
#     vote itself produces 5.
import subprocess, sys

cmd = [
    sys.executable, "-u", "-m", "src.submit",
    "--parent-dir",   str(PARENT_DIR),
    "--model-dir",    str(WEIGHTS_DIR),
    "--output",       str(OUTPUT_CSV),
    "--quantization", "none",   # ship at native bf16 precision (no NF4 compression)
    "--n-samples",    "7",
    "--temperature",  "0.7",
    "--tau",          "0.0",
]
print("Running:", " ".join(cmd), "\n")

# Stream stdout/stderr into the notebook so the TA sees per-question progress.
result = subprocess.run(cmd, cwd=str(PROJECT_DIR), check=False)
if result.returncode != 0:
    raise RuntimeError(f"submit.py exited with code {result.returncode}")

In [ ]:
# === Verify the output ===
import csv

with OUTPUT_CSV.open(encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(f"Wrote {len(rows)} rows to {OUTPUT_CSV}")
print(f"Columns: {list(rows[0].keys()) if rows else '(empty)'}")
print("\nFirst 5 rows:")
for r in rows[:5]:
    print(" ", r)

# Iron-clad invariant: every option in {1,2,3,4,5}
valid = {"1", "2", "3", "4", "5"}
bad = [r for r in rows if str(r["option"]) not in valid]
if bad:
    raise AssertionError(f"{len(bad)} rows have option outside {{1..5}}: {bad[:5]}")
print("\nAll options in {1,2,3,4,5} ✓")

## Troubleshooting

**Missing-path error in cell 1** — check that `PARENT_DIR` points to a directory containing `images/` and `test.csv`. The default assumes the test data sits alongside the notebook.

**Smoke test fails / `submit.py exited with code 1`** — the bundled 7B weights aren't loading correctly. Verify `weights/qwen2.5-vl-7b/` exists and contains `config.json`, `model*.safetensors`, `tokenizer.json`, `preprocessor_config.json`, and `chat_template.json`.

**OOM during inference** — should not happen on the eval L40s (48 GB; bf16 model uses ~15 GB plus a few GB of KV cache). If you're running on a smaller GPU and hit OOM, swap `--quantization none` for `--quantization bnb_nf4` in cell 2 to drop VRAM use to ~5 GB at a small accuracy cost. You can also reduce `--n-samples 7` to `5` or `3` to halve/third the per-question peak.

## References

Open-source projects and papers used in this submission:

- **Qwen2.5-VL** — Bai, S. et al. *Qwen2.5-VL Technical Report*, 2025. <https://github.com/QwenLM/Qwen2.5-VL> (Apache 2.0). The vision-language model that does the actual MCQ reasoning.
- **Self-Consistency Improves Chain of Thought Reasoning in Language Models** — Wang, X. et al., *ICLR 2023*. <https://arxiv.org/abs/2203.11171>. Source of the N-sample majority-vote decoding strategy that drives our +6 score uplift over greedy single-shot.
- **bitsandbytes (NF4 quantization)** — Dettmers, T. et al. *QLoRA: Efficient Finetuning of Quantized LLMs*, NeurIPS 2023. <https://github.com/bitsandbytes-foundation/bitsandbytes>. Provides the optional 4-bit fallback path for smaller GPUs.
- **Hugging Face Transformers** — Wolf, T. et al., *EMNLP 2020 (Demos)*. <https://github.com/huggingface/transformers>. Model loading and `.generate()` infrastructure.
- **qwen-vl-utils** — Qwen team. <https://github.com/QwenLM/Qwen2.5-VL/tree/main/qwen-vl-utils>. Vision-input packing helper used in the chat template.